In [0]:
%run "../utils/03_write_to_delta_utils"

from pyspark.sql.functions import col,coalesce,lit

#---------------------------------------------------
#    Left Join sql_customer, crm_customer Dataframe
#---------------------------------------------------

df_sql_cust = spark.read.table("novamart.silver.sql_customers")
df_crm_cust = spark.read.table("novamart.silver.crm_customers")

df_dim_customers = (df_sql_cust.join(df_crm_cust, "customer_id", "left")
                   .select(
                       col("customer_id"),
                       col("first_name"),
                       col("last_name"),
                       col("full_name"),
                       col("email"),
                       col("cleaned_phone"),
                       col("address"),
                       col("city"),
                       col("region"),
                       col("customer_segment"),
                       col("verified_email"),
                       col("verified_phone"),
                       col("join_timestamp").alias("joined_at"),
                       
                       col("last_compaign_engaged"),
                       coalesce(col("lifetime_value_estimate"), lit(0.0)).alias("lifetime_value_estimate"),
                       coalesce(col("loyalty_tier"), lit("Standard")).alias("loyalty_tier"),
                       coalesce(col("preferred_channel"), lit("Email")).alias("preffered_channel"),
                       col("marketing_opt_in"),
                       col("churn_risk_score"),
                       col("crm_last_updated")
                   ))

#-------------------------------------------
#       Write to delta table dim_customers
#-------------------------------------------

write_delta_table(
    df = df_dim_customers,
    table_name = "novamart.gold.dim_customers",
    write_mode = "merge",
    merge_key = "customer_id",
    cluster_keys = ["customer_id"]
)
